# CLIF - Train ALL Models on Google Colab

**Dataset:** NSL-KDD (Network Security Lab - Knowledge Discovery in Databases)
- **Source:** Canadian Institute for Cybersecurity
- **Train:** 125,973 records | **Test:** 22,544 records
- **Features:** 41 (38 numerical + 3 categorical → 122 after one-hot encoding)
- **Tasks:**
  1. **Binary:** Normal vs Attack (2 classes)
  2. **Multiclass:** Normal, DoS, Probe, R2L, U2R (5 classes)
- **Models:** XGBoost (CUDA), LightGBM, RandomForest, ExtraTrees, GradientBoosting + VotingEnsemble

### Instructions
1. Upload `nsl_kdd_train.txt` and `nsl_kdd_test.txt` from `ai-agents/data/`
2. Run all cells
3. Download all `.joblib` and `.json` files from the output
4. Place them in `ai-agents/models/` on your local machine

In [ ]:
# Install dependencies
!pip install -q xgboost lightgbm scikit-learn imbalanced-learn optuna joblib pandas numpy

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
from datetime import datetime

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, roc_auc_score
)
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    VotingClassifier
)
import xgboost as xgb
import lightgbm as lgb
import optuna
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
CV_FOLDS = 5
N_JOBS = -1

# Check GPU
import subprocess
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout[:500])
    HAS_GPU = True
except:
    HAS_GPU = False
    print('No GPU detected, using CPU')

print(f'XGBoost: {xgb.__version__}, LightGBM: {lgb.__version__}')

In [ ]:
# Upload dataset files
from google.colab import files
import os

if not os.path.exists('nsl_kdd_train.txt'):
    print('Upload nsl_kdd_train.txt and nsl_kdd_test.txt')
    uploaded = files.upload()
else:
    print('Dataset files already present')

In [ ]:
# ============================================================
# CONSTANTS
# ============================================================
FEATURE_COLS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate'
]
ALL_COLS = FEATURE_COLS + ['label', 'difficulty_level']
CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']
NUMERICAL_COLS = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

ATTACK_MAP = {
    'normal': 'Normal',
    'neptune': 'DoS', 'back': 'DoS', 'land': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'mailbomb': 'DoS', 'apache2': 'DoS',
    'processtable': 'DoS', 'udpstorm': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    'mscan': 'Probe', 'saint': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'sendmail': 'R2L', 'named': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'worm': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
    'httptunnel': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R',
}

CLIF_CATEGORY_MAP = {
    'Normal': 'benign',
    'DoS': 'denial_of_service',
    'Probe': 'reconnaissance',
    'R2L': 'remote_access',
    'U2R': 'privilege_escalation',
}

print(f'Features: {len(FEATURE_COLS)}, Categorical: {len(CATEGORICAL_COLS)}, Numerical: {len(NUMERICAL_COLS)}')

In [ ]:
# ============================================================
# LOAD & PREPROCESS
# ============================================================
train = pd.read_csv('nsl_kdd_train.txt', header=None, names=ALL_COLS)
test = pd.read_csv('nsl_kdd_test.txt', header=None, names=ALL_COLS)

print(f'Training: {train.shape[0]:,} rows x {train.shape[1]} cols')
print(f'Testing:  {test.shape[0]:,} rows x {test.shape[1]} cols')

# Binary labels
train['binary_label'] = (train['label'] != 'normal').astype(int)
test['binary_label'] = (test['label'] != 'normal').astype(int)

# Multi-class categories
train['category'] = train['label'].map(ATTACK_MAP).fillna('Unknown')
test['category'] = test['label'].map(ATTACK_MAP).fillna('Unknown')
unknown_mask = test['category'] == 'Unknown'
if unknown_mask.sum() > 0:
    print(f'Unknown labels in test: {unknown_mask.sum()} -> mapped to DoS')
    test.loc[unknown_mask, 'category'] = 'DoS'

print(f"\nBinary: Normal={train['binary_label'].value_counts()[0]:,}, Attack={train['binary_label'].value_counts()[1]:,}")
print(f'\nCategory distribution (train):')
for cat, cnt in train['category'].value_counts().items():
    print(f'  {cat:20s}: {cnt:,}')

In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================
combined = pd.concat([train[FEATURE_COLS], test[FEATURE_COLS]], axis=0, ignore_index=True)
combined_encoded = pd.get_dummies(combined, columns=CATEGORICAL_COLS, drop_first=False)

X_train = combined_encoded.iloc[:len(train)].copy()
X_test = combined_encoded.iloc[len(train):].copy()

scaler = StandardScaler()
X_train[NUMERICAL_COLS] = scaler.fit_transform(X_train[NUMERICAL_COLS])
X_test[NUMERICAL_COLS] = scaler.transform(X_test[NUMERICAL_COLS])

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

feature_names = list(X_train.columns)
all_encoded_cols = X_train.columns.tolist()

y_train_bin = train['binary_label'].values
y_test_bin = test['binary_label'].values

le_multi = LabelEncoder()
y_train_multi = le_multi.fit_transform(train['category'].values)
y_test_multi = le_multi.transform(test['category'].values)

X_train_np = X_train.values
X_test_np = X_test.values

print(f'Features after encoding: {len(feature_names)}')
print(f'Multi-class labels: {list(le_multi.classes_)}')
print(f'X_train: {X_train_np.shape}, X_test: {X_test_np.shape}')

In [ ]:
# ============================================================
# SMOTE
# ============================================================
def apply_smote(X, y, name=''):
    unique, counts = np.unique(y, return_counts=True)
    min_count = counts.min()
    if counts.max() / min_count > 3:
        k = min(5, min_count - 1) if min_count > 1 else 1
        smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
        X_r, y_r = smote.fit_resample(X, y)
        print(f'SMOTE {name}: {len(X):,} -> {len(X_r):,}')
        return X_r, y_r
    print(f'SMOTE {name}: balanced, skipping')
    return X, y

X_train_bin_sm, y_train_bin_sm = apply_smote(X_train_np, y_train_bin, 'binary')
X_train_multi_sm, y_train_multi_sm = apply_smote(X_train_np, y_train_multi, 'multiclass')

In [ ]:
# ============================================================
# OPTUNA OBJECTIVES
# ============================================================
def xgb_objective(trial, X, y, n_classes, cv):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': RANDOM_STATE, 'n_jobs': N_JOBS,
        'tree_method': 'hist', 'device': 'cuda' if HAS_GPU else 'cpu',
    }
    if n_classes == 2:
        params.update({'objective': 'binary:logistic', 'eval_metric': 'logloss'})
    else:
        params.update({'objective': 'multi:softmax', 'num_class': n_classes, 'eval_metric': 'mlogloss'})
    model = xgb.XGBClassifier(**params)
    return cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=1).mean()

def lgb_objective(trial, X, y, n_classes, cv):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 16, 256),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': RANDOM_STATE, 'n_jobs': N_JOBS, 'verbose': -1,
    }
    if n_classes == 2:
        model = lgb.LGBMClassifier(**params, objective='binary')
    else:
        model = lgb.LGBMClassifier(**params, objective='multiclass', num_class=n_classes)
    return cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=1).mean()

def rf_objective(trial, X, y, n_classes, cv):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap': True, 'random_state': RANDOM_STATE, 'n_jobs': N_JOBS,
    }
    return cross_val_score(RandomForestClassifier(**params), X, y, cv=cv, scoring='accuracy', n_jobs=1).mean()

def et_objective(trial, X, y, n_classes, cv):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': RANDOM_STATE, 'n_jobs': N_JOBS,
    }
    return cross_val_score(ExtraTreesClassifier(**params), X, y, cv=cv, scoring='accuracy', n_jobs=1).mean()

def gb_objective(trial, X, y, n_classes, cv):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': RANDOM_STATE,
    }
    return cross_val_score(GradientBoostingClassifier(**params), X, y, cv=cv, scoring='accuracy', n_jobs=1).mean()

print('Objectives defined.')

In [ ]:
# ============================================================
# TRAINING FUNCTIONS
# ============================================================
def train_and_evaluate(name, objective_fn, X_train, y_train, X_test, y_test,
                       n_classes, label_names, n_trials=50):
    """Train with Optuna HPO and evaluate."""
    print(f'\n{"="*60}')
    print(f'Training {name} ({n_trials} trials)')
    print(f'{"="*60}')

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    study = optuna.create_study(direction='maximize')
    study.optimize(
        lambda trial: objective_fn(trial, X_train, y_train, n_classes, cv),
        n_trials=n_trials, show_progress_bar=True,
        catch=(ValueError, Exception),
    )

    best_params = study.best_params
    print(f'Best CV: {study.best_value:.4f}')
    print(f'Params: {json.dumps({k: round(v,4) if isinstance(v,float) else v for k,v in best_params.items()}, indent=2)}')

    # Build final model
    if 'XGBoost' in name:
        fp = {**best_params, 'random_state': RANDOM_STATE, 'n_jobs': N_JOBS,
              'tree_method': 'hist', 'device': 'cuda' if HAS_GPU else 'cpu'}
        if n_classes == 2:
            fp.update({'objective': 'binary:logistic', 'eval_metric': 'logloss'})
        else:
            fp.update({'objective': 'multi:softmax', 'num_class': n_classes, 'eval_metric': 'mlogloss'})
        model = xgb.XGBClassifier(**fp)
    elif 'LightGBM' in name:
        fp = {**best_params, 'random_state': RANDOM_STATE, 'n_jobs': N_JOBS, 'verbose': -1}
        if n_classes == 2:
            model = lgb.LGBMClassifier(**fp, objective='binary')
        else:
            model = lgb.LGBMClassifier(**fp, objective='multiclass', num_class=n_classes)
    elif 'RandomForest' in name:
        model = RandomForestClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=N_JOBS, bootstrap=True)
    elif 'ExtraTrees' in name:
        model = ExtraTreesClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=N_JOBS)
    elif 'GradientBoosting' in name:
        model = GradientBoostingClassifier(**best_params, random_state=RANDOM_STATE)
    else:
        raise ValueError(f'Unknown: {name}')

    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = model.predict(X_test)
    inference_time = (time.time() - t0) / len(X_test) * 1000

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f'Test Acc: {acc:.4f}, F1: {f1:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}')
    print(f'Train: {train_time:.1f}s, Inference: {inference_time:.4f} ms/sample')
    print(classification_report(y_test, y_pred, target_names=label_names))

    auc = None
    if n_classes == 2 and hasattr(model, 'predict_proba'):
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        print(f'AUC-ROC: {auc:.4f}')

    return {
        'name': name, 'model': model, 'best_params': best_params,
        'cv_accuracy': study.best_value, 'test_accuracy': acc,
        'test_precision': prec, 'test_recall': rec, 'test_f1': f1,
        'auc_roc': auc, 'train_time': train_time, 'inference_ms': inference_time,
    }

def build_ensemble(results, X_train, y_train, X_test, y_test, n_classes, label_names):
    """Voting ensemble from top 3."""
    top3 = sorted(results, key=lambda r: r['cv_accuracy'], reverse=True)[:3]
    estimators = [(r['name'].replace(' ', '_'), r['model']) for r in top3]
    print(f'\nEnsemble components: {[e[0] for e in estimators]}')

    ensemble = VotingClassifier(estimators=estimators, voting='soft', n_jobs=1)
    t0 = time.time()
    ensemble.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = ensemble.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f'Ensemble Acc: {acc:.4f}, F1: {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=label_names))

    auc = None
    if n_classes == 2 and hasattr(ensemble, 'predict_proba'):
        auc = roc_auc_score(y_test, ensemble.predict_proba(X_test)[:, 1])
        print(f'AUC-ROC: {auc:.4f}')

    return {
        'name': 'VotingEnsemble', 'model': ensemble, 'best_params': {},
        'cv_accuracy': np.mean([r['cv_accuracy'] for r in top3]),
        'test_accuracy': acc, 'test_precision': prec, 'test_recall': rec,
        'test_f1': f1, 'auc_roc': auc, 'train_time': train_time, 'inference_ms': 0,
    }

print('Training functions ready.')

## Binary Classification Training
XGBoost (50 trials, GPU) → LightGBM (50) → RandomForest (20) → ExtraTrees (20) → GradientBoosting (20) → Ensemble

In [ ]:
# ============================================================
# BINARY TRAINING
# ============================================================
binary_labels = ['Normal', 'Attack']
binary_configs = [
    ('XGBoost (binary)', xgb_objective, 50),
    ('LightGBM (binary)', lgb_objective, 50),
    ('RandomForest (binary)', rf_objective, 20),
    ('ExtraTrees (binary)', et_objective, 20),
    ('GradientBoosting (binary)', gb_objective, 20),
]

results_binary = []
for name, obj_fn, n_trials in binary_configs:
    result = train_and_evaluate(
        name, obj_fn, X_train_bin_sm, y_train_bin_sm,
        X_test_np, y_test_bin, 2, binary_labels, n_trials
    )
    results_binary.append(result)
    # Save immediately
    safe = name.split('(')[0].strip().lower().replace(' ', '')
    joblib.dump(result['model'], f'{safe}_binary.joblib')
    print(f'  [SAVED] {safe}_binary.joblib')

# Build ensemble
ens_bin = build_ensemble(results_binary, X_train_bin_sm, y_train_bin_sm,
                         X_test_np, y_test_bin, 2, binary_labels)
results_binary.append(ens_bin)
joblib.dump(ens_bin['model'], 'votingensemble_binary.joblib')
print('  [SAVED] votingensemble_binary.joblib')

## Multiclass Classification Training
Same models, 5 classes: Normal, DoS, Probe, R2L, U2R

In [ ]:
# ============================================================
# MULTICLASS TRAINING
# ============================================================
multi_labels = list(le_multi.classes_)
n_multi = len(multi_labels)

multi_configs = [
    ('XGBoost (multiclass)', xgb_objective, 50),
    ('LightGBM (multiclass)', lgb_objective, 50),
    ('RandomForest (multiclass)', rf_objective, 20),
    ('ExtraTrees (multiclass)', et_objective, 20),
    ('GradientBoosting (multiclass)', gb_objective, 20),
]

results_multi = []
for name, obj_fn, n_trials in multi_configs:
    result = train_and_evaluate(
        name, obj_fn, X_train_multi_sm, y_train_multi_sm,
        X_test_np, y_test_multi, n_multi, multi_labels, n_trials
    )
    results_multi.append(result)
    safe = name.split('(')[0].strip().lower().replace(' ', '')
    joblib.dump(result['model'], f'{safe}_multiclass.joblib')
    print(f'  [SAVED] {safe}_multiclass.joblib')

# Build ensemble
ens_multi = build_ensemble(results_multi, X_train_multi_sm, y_train_multi_sm,
                           X_test_np, y_test_multi, n_multi, multi_labels)
results_multi.append(ens_multi)
joblib.dump(ens_multi['model'], 'votingensemble_multiclass.joblib')
print('  [SAVED] votingensemble_multiclass.joblib')

In [ ]:
# ============================================================
# SAVE FINAL ARTIFACTS
# ============================================================
best_binary = max(results_binary, key=lambda r: r['test_f1'])
best_multi = max(results_multi, key=lambda r: r['test_f1'])

# Save best models as the primary classifiers
joblib.dump(best_binary['model'], 'binary_classifier.joblib')
joblib.dump(best_multi['model'], 'multiclass_classifier.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(le_multi, 'label_encoder.joblib')

config = {
    'version': '1.0.0',
    'created': datetime.now().isoformat(),
    'dataset': 'NSL-KDD',
    'feature_cols': FEATURE_COLS,
    'categorical_cols': CATEGORICAL_COLS,
    'numerical_cols': NUMERICAL_COLS,
    'encoded_feature_names': all_encoded_cols,
    'binary_model': {
        'name': best_binary['name'],
        'file': 'binary_classifier.joblib',
        'accuracy': best_binary['test_accuracy'],
        'f1': best_binary['test_f1'],
        'precision': best_binary['test_precision'],
        'recall': best_binary['test_recall'],
        'auc_roc': best_binary['auc_roc'],
        'params': {k: (round(v,6) if isinstance(v,float) else v) for k,v in best_binary['best_params'].items()},
    },
    'multiclass_model': {
        'name': best_multi['name'],
        'file': 'multiclass_classifier.joblib',
        'classes': list(le_multi.classes_),
        'clif_category_map': CLIF_CATEGORY_MAP,
        'accuracy': best_multi['test_accuracy'],
        'f1': best_multi['test_f1'],
        'precision': best_multi['test_precision'],
        'recall': best_multi['test_recall'],
        'params': {k: (round(v,6) if isinstance(v,float) else v) for k,v in best_multi['best_params'].items()},
    },
    'attack_map': ATTACK_MAP,
}

with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

leaderboard = {
    'binary': [{k:v for k,v in r.items() if k != 'model'}
               for r in sorted(results_binary, key=lambda r: r['test_f1'], reverse=True)],
    'multiclass': [{k:v for k,v in r.items() if k != 'model'}
                   for r in sorted(results_multi, key=lambda r: r['test_f1'], reverse=True)],
}
with open('leaderboard.json', 'w') as f:
    json.dump(leaderboard, f, indent=2)

print(f'\n{"="*60}')
print(f'BEST Binary:     {best_binary["name"]} -> Acc={best_binary["test_accuracy"]:.4f}, F1={best_binary["test_f1"]:.4f}')
print(f'BEST Multiclass: {best_multi["name"]} -> Acc={best_multi["test_accuracy"]:.4f}, F1={best_multi["test_f1"]:.4f}')
print(f'{"="*60}')

# List all saved files
import glob
for f in sorted(glob.glob('*.joblib') + glob.glob('*.json')):
    print(f'  {f} ({os.path.getsize(f)/1024:.0f} KB)')

In [ ]:
# ============================================================
# DOWNLOAD ALL FILES
# ============================================================
from google.colab import files
import glob

for f in sorted(glob.glob('*.joblib') + glob.glob('*.json')):
    print(f'Downloading {f}...')
    files.download(f)

print('\nDone! Place all downloaded files in ai-agents/models/ on your local machine.')

## Final Leaderboard

In [ ]:
print(f'{"Binary Classification":^60}')
print(f'{"Model":<35} {"CV":>6} {"Acc":>6} {"F1":>6} {"AUC":>6}')
print('-'*60)
for r in sorted(results_binary, key=lambda x: x['test_f1'], reverse=True):
    auc_str = f"{r['auc_roc']:.4f}" if r['auc_roc'] else 'N/A'
    star = ' *' if r['name'] == best_binary['name'] else ''
    print(f"{r['name']:<35} {r['cv_accuracy']:.4f} {r['test_accuracy']:.4f} {r['test_f1']:.4f} {auc_str}{star}")

print(f'\n{"Multiclass Classification":^60}')
print(f'{"Model":<35} {"CV":>6} {"Acc":>6} {"F1":>6}')
print('-'*60)
for r in sorted(results_multi, key=lambda x: x['test_f1'], reverse=True):
    star = ' *' if r['name'] == best_multi['name'] else ''
    print(f"{r['name']:<35} {r['cv_accuracy']:.4f} {r['test_accuracy']:.4f} {r['test_f1']:.4f}{star}")

print('\n* = Best model (saved as primary classifier)')